# 연습문제: 임베딩으로 문서 검색하기

KURE-v2의 다중 벡터 임베딩과 MaxSim 유사도를 사용해 한국어 문서를 검색합니다. 셀을 위에서 아래 순서로 실행하세요.

In [ ]:
# Colab에서는 먼저 이 셀을 실행하세요.
!pip install -q 'sentence-transformers>=6.0,<7.0'

from sentence_transformers import MultiVectorEncoder
from kure_search import search

MODEL_NAME = 'nlpai-lab/KURE-v2'
model = MultiVectorEncoder(MODEL_NAME)

documents = [
    '세종대왕은 1443년에 훈민정음을 창제하고 1446년에 반포했다.',
    '이순신 장군은 조선 시대 일본의 침략에 맞서 뛰어난 전략으로 싸웠다.',
    '고려는 몽골의 침입에 맞서 수도를 강화도로 옮겼다.',
    '프랑스의 수도는 파리이고 에펠탑으로 유명하다.',
    '김치는 배추를 소금에 절인 뒤 양념을 넣어 발효시킨 한국 전통 음식이다.',
    '비빔밥은 밥 위에 나물과 고추장을 넣어 비벼 먹는 한국 음식이다.',
    '태권도는 한국 전통 무술로 발차기와 손기술을 사용한다.',
    '떡볶이는 쌀떡과 어묵을 매콤한 양념에 졸여 먹는 음식이다.',
    '광합성은 식물이 빛 에너지를 이용해 물과 이산화탄소로 포도당을 만드는 과정이다.',
    '지구는 태양 주위를 약 365일에 한 바퀴 돈다.',
    '물은 1기압에서 섭씨 100도에 끓는다.',
    '백신은 몸의 면역 체계가 병원체를 미리 기억하도록 돕는다.',
    '축구 경기는 전반과 후반 각각 45분으로 진행된다.',
    '마라톤의 공식 거리는 42.195킬로미터이다.',
    '스키는 눈 덮인 슬로프에서 스키를 타고 내려오는 스포츠다.',
    '서울은 대한민국의 수도이고 한강이 도시를 가로지른다.',
    '제주도는 화산 활동으로 만들어진 섬으로 한라산이 있다.',
    '제주도는 우리나라에서 가장 큰 섬이다.',
    '경주에는 불국사와 첨성대 같은 신라 유적이 많다.',
    '부산은 우리나라 최대 항구 도시이다.'
]

In [ ]:
# 문제 1. 문서 색인 만들기
doc_embeddings = model.encode_document(documents, show_progress_bar=True)

print(f'문서 개수: {len(documents)}')
for index in (0, 5, 10):
    print(f'문서 {index} 임베딩 형태: {doc_embeddings[index].shape}')

In [ ]:
# 문제 2. 상위 3개 문서 검색하기
def print_results(query, k=3):
    print(f'\n질문: {query}')
    for rank, (index, score, document) in enumerate(
        search(query, k, model, documents, doc_embeddings), start=1
    ):
        print(f'{rank}위 | 점수 {score:.4f} | 문서 {index}: {document}')

print_results('훈민정음은 언제 만들어졌어요?')

In [ ]:
# 문제 3. 단어가 겹치지 않는 질문으로 검색하기
queries = [
    '세종이 만든 글자의 이름은 무엇인가요?',
    '발차기 위주의 한국 무술',
    '햇빛과 이산화탄소로 양분을 만드는 생물의 작용',
    '가장 긴 달리기 종목의 거리',
    '항구가 발달한 우리나라의 도시',
    '면역 반응을 미리 학습시키는 예방 방법',
    '화산섬에서 가장 높은 산은 무엇인가요?',
    '떡과 어묵을 매운 소스에 조린 음식'
]

for query in queries:
    print_results(query)

In [ ]:
# 문제 4. 검색 정확도 측정하기
test_set = [
    ('세종이 만든 한글의 이름은?', 0),
    ('거북선을 이끈 장수는 누구인가?', 1),
    ('쌀떡에 매콤한 양념으로 만든 요리', 7),
    ('태권도의 몸동작과 발차기', 6),
    ('이산화탄소와 빛으로 양분을 만드는 원리', 8),
    ('예방접종은 어떤 작용을 하나요?', 11),
    ('가장 긴 달리기 종목의 거리', 13),
    ('우리나라에서 가장 큰 섬', 17),
    ('천년 고도의 유적지', 18),
    ('항구가 가장 많이 발달한 도시', 19),
]

correct = 0
for query, expected_index in test_set:
    top3 = search(query, 3, model, documents, doc_embeddings)
    predicted_index = top3[0][0]
    is_correct = predicted_index == expected_index
    correct += is_correct
    print(
        f"{'O' if is_correct else 'X'} | 질문: {query} | "
        f'예측: {predicted_index}, 정답: {expected_index}'
    )
    if not is_correct:
        print('  Top-3:', [(index, round(score, 4)) for index, score, _ in top3])

accuracy = correct / len(test_set)
print(f'\nTop-1 정확도: {accuracy:.1%} ({correct}/{len(test_set)})')